# Batch Integration

Low-level 1-D and 2-D integration reference. The smoke path makes a
compact synthetic pattern; real mode uses an explicit image and PONI.
For a durable multi-frame reduction, continue to notebook 06.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from xrd_tools.core.containers import IntegrationResult1D
from xrd_tools.integrate import integrate_1d, integrate_2d, load_poni
from xrd_tools.io import read_image
from xrd_tools.viz import plot_1d


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
image_file = TEST_DATA / "image.tif"
poni_file = TEST_DATA / "calibration.poni"
npt_1d = 256
npt_2d = (256, 72)
run_button = widgets.Button(description="Run integration", button_style="primary")
status = widgets.HTML("<i>Configure real paths, then run explicitly.</i>")
display(widgets.VBox([widgets.HBox([run_button]), status]))


In [ ]:
if SMOKE_MODE:
    q = np.linspace(1.0, 4.0, npt_1d)
    intensity = 20 + 90 * np.exp(-0.5 * ((q - 2.35) / 0.07) ** 2)
    result_1d = IntegrationResult1D(q, intensity, unit="q_A^-1")
else:
    assert image_file.is_file(), f"Missing detector image: {image_file}"
    assert poni_file.is_file(), f"Missing PONI calibration: {poni_file}"
    result_1d = integrate_1d(read_image(image_file), load_poni(poni_file), npt=npt_1d)

fig, ax = plt.subplots(figsize=(7, 3))
plot_1d(ax, result_1d.radial, result_1d.intensity, fmt="-", attrs={"xlabel": f"q ({result_1d.unit})", "ylabel": "Intensity"})
plt.show()
